# Titanic Passenger Survival - Exploratory Data Analysis

This notebook explores the Kaggle Titanic training dataset using Pandas. It covers data loading, missing values, cleaning, summary statistics, and survival comparisons.

## 1. Load the dataset

In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw/train.csv")
if not data_path.exists():
    data_path = Path("data/raw/train.csv")

In [2]:
df = pd.read_csv(data_path)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 891
Columns: 12


## 2. Inspect the data

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [5]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [7]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate passenger IDs:", df["PassengerId"].duplicated().sum())

Duplicate rows: 0
Duplicate passenger IDs: 0


## 3. Clean the data

In [8]:
df_clean = df.copy()

df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())
df_clean["Embarked"] = df_clean["Embarked"].fillna(
    df_clean["Embarked"].mode()[0]
)
df_clean = df_clean.drop(columns="Cabin")

In [9]:
print("Remaining missing values:", df_clean.isna().sum().sum())
df_clean.info()

Remaining missing values: 0
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


## 4. Overall survival

In [10]:
survival_counts = df_clean["Survived"].value_counts().sort_index()
survival_counts.index = ["Did not survive", "Survived"]
survival_counts

Did not survive    549
Survived           342
Name: count, dtype: int64

In [11]:
overall_survival_rate = df_clean["Survived"].mean() * 100
print(f"Overall survival rate: {overall_survival_rate:.2f}%")

Overall survival rate: 38.38%


## 5. Survival by passenger characteristics

In [12]:
survival_by_sex = df_clean.groupby("Sex", observed=True)["Survived"].mean() * 100
survival_by_sex.round(2)

Sex
female    74.20
male      18.89
Name: Survived, dtype: float64

In [13]:
survival_by_class = df_clean.groupby("Pclass")["Survived"].mean() * 100
survival_by_class.round(2)

Pclass
1    62.96
2    47.28
3    24.24
Name: Survived, dtype: float64

In [14]:
survival_by_port = df_clean.groupby("Embarked", observed=True)["Survived"].mean() * 100
survival_by_port.round(2)

Embarked
C    55.36
Q    38.96
S    33.90
Name: Survived, dtype: float64

## 6. Age and survival

In [15]:
age_bins = [0, 12, 18, 35, 60, 100]
age_labels = ["Child", "Teenager", "Young adult", "Middle-aged", "Senior"]

df_clean["AgeGroup"] = pd.cut(
    df_clean["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

survival_by_age = df_clean.groupby("AgeGroup", observed=True)["Survived"].mean() * 100
survival_by_age.round(2)

AgeGroup
Child          57.97
Teenager       42.86
Young adult    35.33
Middle-aged    40.00
Senior         22.73
Name: Survived, dtype: float64

## 7. Numerical summary

In [16]:
numerical_columns = [
    "Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"
]
df_clean[numerical_columns].corr().round(2)

,Survived,Pclass,Age,SibSp,Parch,Fare
Survived,1.00,-0.34,-0.06,-0.04,0.08,0.26
Pclass,-0.34,1.00,-0.34,0.08,0.02,-0.55
Age,-0.06,-0.34,1.00,-0.23,-0.17,0.10
SibSp,-0.04,0.08,-0.23,1.00,0.41,0.16
Parch,0.08,0.02,-0.17,0.41,1.00,0.22
Fare,0.26,-0.55,0.10,0.16,0.22,1.00


## 8. Save the cleaned dataset

In [17]:
output_path = Path("../data/processed/titanic_cleaned.csv")
if not output_path.parent.exists():
    output_path = Path("data/processed/titanic_cleaned.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False)
print(output_path)

..\data\processed\titanic_cleaned.csv
